# 2026-10-17_2 scTenifoldKnk 胚胎 MG target 配对参数表与扰动可视化

基于 `2026-10-17_1` 胚胎 MG ONECUT1 stage 拆分 KO 输出（Early / Mid / Late × central(macula) / peripheral），输出关注配对的完整参数表（CSV）与百分位柱状图：

- ONECUT1 (MG) → ABCB11、TTC39B：各 6 组（4 组有 KO 结果：E-c / M-c / L-c / L-p；E-p / M-p 因 qc_cells < 500 KO 未运行，显式 NA 行）
- MAFK (MG) → TFRC：6 组全部显式 NA 行——MAFK 在胚胎 MG 中不可行（检出率 0–2.4%，仅 Early-central 2.40% 勉强过 0.01 且仅 501 cells，经用户决定本轮不运行）

阈值口径：E-c / M-c 为官方 0.05；L-c / L-p 经用户决定放宽 0.01（ONECUT1 检出率 3.89% / 3.45%）；不同口径网络间 FC / 排名 / 百分位不可直接比较；胚胎与成年为不同数据集，任何数值不可跨数据集比较。

列定义（与 2026-10-16_3 成年 CSV 完全一致）：`network_threshold` = QC 阈值口径；`n_tested` = 网络内 tested genes 数（排除 TF 自身）；`distance` / `Z` / `FC` / `p.value` / `p.adj` = diffRegulation 原始参数；`FC_rank` / `percentile` = FC 降序排名与百分位（100 = 最强扰动）；`significant` = p.adj < 0.05；`direction` / `directionScore` = KO 预测方向（与上游 DEG 方向是两回事，勿混用）。

输出：`Data/2026-10-17_dev_target_pairs_params.csv`（18 行）+ `Plot/2026-10-17_Plot1_dev_target_pairs_perturbation_percentile.pdf`（仅 PDF）。

In [ ]:
# ---- 1. 配置：路径 / 配对 / 阈值口径（集中修改处） ----------------------------
suppressPackageStartupMessages({
  library(data.table)
})
set.seed(1)

KNK_DIR    <- "/project/work/2026-10-13_scTenifoldKnk/Data/"
DATE       <- "2026-10-17"
OUT_CSV    <- file.path(KNK_DIR, paste0(DATE, "_dev_target_pairs_params.csv"))
PADJ_CUTOFF <- 0.05

# 关注的 TF-target 配对（胚胎轮；MAFK 无 KO，全部显式 NA 行）
pair_base <- data.table(
  TF          = c("ONECUT1", "ONECUT1", "MAFK"),
  target_gene = c("ABCB11", "TTC39B", "TFRC"),
  celltype    = c("MG", "MG", "MG")
)
group_grid <- CJ(region = c("central", "peripheral"), stage = c("Early", "Mid", "Late"))
TARGET_PAIRS <- rbindlist(lapply(seq_len(nrow(pair_base)), function(i) {
  cbind(pair_base[i, .(TF, target_gene, celltype)], group_grid)
}))

# 各 KO 网络的 QC 阈值口径（Late 期放宽 0.01；MAFK / Early-peripheral / Mid-peripheral 无网络 -> 不标）
NETWORK_MINPCT <- rbindlist(lapply("ONECUT1", function(tf) {
  group_grid[, .(TF = tf, region, stage, network_threshold = "0.05")]
}))
NETWORK_MINPCT[TF == "ONECUT1" & stage == "Late", network_threshold := "0.01"]

# NA 行原因（KO 未运行的组合）：MAFK 全部 6 组 + ONECUT1 E-peripheral/M-peripheral
NA_REASON <- rbind(
  CJ(TF = "MAFK", region = c("central", "peripheral"), stage = c("Early", "Mid", "Late")
     )[, na_reason := "KO not run: MAFK infeasible in embryonic MG (detection 0-2.4%; only Early-central 2.40% passes 0.01 with only 501 cells)"],
  data.table(TF = "ONECUT1", region = "peripheral", stage = c("Early", "Mid"),
             na_reason = "KO not run: qc_cells < 500 (Early 24 / Mid 137 cells)")
)

cat("配对 x 组合行数:", nrow(TARGET_PAIRS), "\n")
print(TARGET_PAIRS)

In [ ]:
# ---- 2. 读取各组合 diffRegulation，计算 target 扰动参数并输出 CSV --------------
## 文件发现不限日期前缀；多文件取最新；文件缺失或 target 不在检验基因集 -> NA 行 + na_reason
pair_res <- rbindlist(lapply(seq_len(nrow(TARGET_PAIRS)), function(i) {
  tp <- TARGET_PAIRS[i]
  nr <- NA_REASON[TF == tp$TF & region == tp$region & stage == tp$stage, na_reason]
  reason <- if (length(nr)) nr else NA_character_
  f <- list.files(KNK_DIR, pattern = paste0("_dev_", tp$celltype, "_", tp$region, "_",
                                            tp$stage, "_", tp$TF, "_diffRegulation[.]tsv$"),
                  full.names = TRUE)
  if (length(f) > 1) {
    warning(sprintf("%s (%s, %s, %s) 匹配到多个文件，使用最新: %s",
                    tp$TF, tp$celltype, tp$region, tp$stage, basename(sort(f)[length(f)])))
    f <- sort(f)[length(f)]
  }
  d <- NULL
  if (length(f) == 1) {
    d <- fread(f)
    miss <- setdiff(c("gene", "distance", "Z", "FC", "p.value", "p.adj",
                      "direction", "directionScore"), names(d))
    if (length(miss) > 0) stop(basename(f), " 缺少必需列: ", paste(miss, collapse = ", "))
  } else if (is.na(reason)) {
    reason <- "KO not run: no diffRegulation file for this combination"
  }
  ok <- !is.null(d) && tp$target_gene %in% d$gene
  if (!is.null(d) && !ok) {
    reason <- "target gene not in tested genes of this network"
  }
  if (!ok) {
    out <- data.table(network_threshold = NA_character_, n_tested = NA_integer_,
                      distance = NA_real_, Z = NA_real_, FC = NA_real_,
                      p.value = NA_real_, p.adj = NA_real_,
                      FC_rank = NA_integer_, percentile = NA_real_,
                      significant = NA, direction = NA_character_,
                      directionScore = NA_real_, na_reason = reason)
  } else {
    dd <- d[gene != tp$TF]   # 排除被敲除的 TF 自身
    rk <- frank(-dd$FC, ties.method = "first")
    pct <- round(100 * (length(rk) - rk) / pmax(length(rk) - 1, 1), 1)
    j <- which(dd$gene == tp$target_gene)
    thr <- NETWORK_MINPCT[TF == tp$TF & region == tp$region & stage == tp$stage, network_threshold]
    out <- data.table(network_threshold = if (length(thr)) thr else NA_character_,
                      n_tested = length(rk),
                      distance = dd$distance[j], Z = dd$Z[j], FC = dd$FC[j],
                      p.value = dd$p.value[j], p.adj = dd$p.adj[j],
                      FC_rank = as.integer(rk[j]), percentile = pct[j],
                      significant = dd$p.adj[j] < PADJ_CUTOFF,
                      direction = dd$direction[j],
                      directionScore = dd$directionScore[j],
                      na_reason = reason)
  }
  cbind(tp, out)
}))

setcolorder(pair_res, c("TF", "target_gene", "celltype", "region", "stage",
                        "network_threshold", "n_tested", "distance", "Z", "FC",
                        "p.value", "p.adj", "FC_rank", "percentile", "significant",
                        "direction", "directionScore", "na_reason"))
pair_res <- pair_res[order(match(TF, unique(pair_base$TF)), target_gene,
                           match(region, c("central", "peripheral")),
                           match(stage, c("Early", "Mid", "Late")))]

fwrite(pair_res, OUT_CSV)
cat("写出:", OUT_CSV, "\n")
print(pair_res, nrows = 50)

In [ ]:
# ---- 3. 验证：行数 / NA 统计 / CSV 回读 ---------------------------------------
stopifnot(nrow(pair_res) == 18)
stopifnot(ncol(pair_res) == 18)
## 8 行有数值（ONECUT1 2 配对 x 4 组），10 行 NA（MAFK 6 行 + ONECUT1 E-p/M-p 4 行）
stopifnot(sum(!is.na(pair_res$percentile)) == 8)
stopifnot(sum(is.na(pair_res$percentile)) == 10)
## 所有 NA 行均有 na_reason
stopifnot(all(!is.na(pair_res[is.na(percentile), na_reason])))

csv_back <- fread(OUT_CSV)
stopifnot(nrow(csv_back) == 18, ncol(csv_back) == 18,
          identical(csv_back$TF, pair_res$TF),
          identical(csv_back$target_gene, pair_res$target_gene))
cat("验证通过：18 行（8 数据行 + 10 NA 行）、18 列\n")
print(csv_back[!is.na(percentile), .(TF, target_gene, region, stage, network_threshold,
                                      n_tested, percentile, significant, p.adj)])

In [ ]:
# ---- 4. Plot: target 扰动百分位柱状图（分面 TF x Region，x = target）----------
suppressPackageStartupMessages(library(ggplot2))
# 胚胎 stage 配色（沿用 /project/work/color.ipynb：cols.stage 三色）
cols.devstage <- c(Early = "#59A14F", Mid = "#EDC948", Late = "#B07AA1")
region_labels <- c(central = "Central KO (macula)", peripheral = "Peripheral KO")

plot_dt <- copy(pair_res[!is.na(percentile)])
plot_dt[, Region := factor(region_labels[region], levels = unname(region_labels))]
plot_dt[, Stage := factor(stage, levels = c("Early", "Mid", "Late"))]
plot_dt[, TF := factor(TF, levels = unique(pair_base$TF))]

p <- ggplot(plot_dt, aes(target_gene, percentile, fill = Stage)) +
  geom_col(position = position_dodge(0.75), width = 0.65, color = "grey20", linewidth = 0.2) +
  geom_text(aes(label = sprintf("%.1f", percentile)),
            position = position_dodge(0.75), vjust = -0.5, size = 3) +
  scale_fill_manual(values = cols.devstage, name = NULL) +
  scale_y_continuous(limits = c(0, 100), expand = expansion(mult = c(0.02, 0.12))) +
  facet_grid(TF ~ Region, scales = "free_x", space = "free_x") +
  labs(title = "Network perturbation percentile of target genes after ONECUT1 virtual knockout (embryonic MG)",
       subtitle = "Percentile ranked by FC among all tested genes within each KO network; central = macula; Late networks at relaxed minPCT 0.01, others 0.05; MAFK and E-periph/M-periph KO not run (NA)",
       x = "Target gene", y = "Perturbation percentile (%)") +
  theme_bw(base_size = 10) +
  theme(legend.position = "top",
        axis.text.x = element_text(size = 9),
        strip.text = element_text(size = 8.5))
print(p)
ggsave(file.path("/project/work/2026-10-13_scTenifoldKnk/Plot/",
                 paste0(DATE, "_Plot1_dev_target_pairs_perturbation_percentile.pdf")),
       p, width = 8, height = 4.5)